# 📊 EDA — Nigerian Treasury Bill Predictive Model
**Data Source:** CBN Primary Market Auction Data (2002–2026)  
**File:** `Primary_Market_in_Excel.xlsx`  
**Author:** Fortlytics  
**Purpose:** Exploratory Data Analysis before feature engineering and model training

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import re

warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
COLORS = {'91': '#1f77b4', '182': '#ff7f0e', '364': '#2ca02c'}

print('Libraries loaded ✓')

## 2. Load Raw Data

In [ ]:
raw = pd.read_excel('../data/raw/Primary_Market_in_Excel.xlsx')
raw['auctionDate'] = pd.to_datetime(raw['auctionDate'], format='mixed', dayfirst=False)
raw = raw.sort_values('auctionDate').reset_index(drop=True)

print(f'Shape: {raw.shape}')
print(f'Date range: {raw["auctionDate"].min().date()} → {raw["auctionDate"].max().date()}')
print(f'\nColumns:\n{raw.dtypes}')
raw.head()

## 3. Data Quality Check

In [ ]:
print('=== Missing Values ===')
miss = raw.isnull().sum()
miss_pct = (miss / len(raw) * 100).round(2)
print(pd.DataFrame({'Missing': miss, 'Pct (%)': miss_pct})[miss > 0])

print(f'\n=== Unique Tenor Values ({raw["tenor"].nunique()} variants) ===')
print(raw['tenor'].value_counts().to_string())

print(f'\n=== trueYield == 0 rows: {(raw["trueYield"]==0).sum()} of {len(raw)} ===')
print('(Recent auctions use `rate` column instead)')

## 4. Clean & Normalise

In [ ]:
def normalise_tenor(t):
    key = str(t).strip().lower()
    match = re.match(r'^(\d+)', key)
    if match:
        n = int(match.group(1))
        if 88  <= n <= 95:  return 91
        if 178 <= n <= 185: return 182
        if 340 <= n <= 366: return 364
    return None

df = raw.copy()
df['tenor_days'] = df['tenor'].apply(normalise_tenor)
df = df.dropna(subset=['tenor_days'])
df['tenor_days'] = df['tenor_days'].astype(int)
df = df[df['tenor_days'].isin([91, 182, 364])]

# Resolve yield: use trueYield where available, else rate
df['yield_pct'] = np.where(df['trueYield'] > 0, df['trueYield'], df['rate'])
df = df[df['yield_pct'] > 0]

# Demand ratios
df['subscription_ratio'] = df['totalSubscription'] / df['amtOffered'].replace(0, np.nan)
df['allotment_ratio']    = df['totalSuccessful']    / df['totalSubscription'].replace(0, np.nan)

df['tenor_label'] = df['tenor_days'].astype(str) + '-Day'

df = df.sort_values(['auctionDate','tenor_days']).reset_index(drop=True)
df.to_csv('../data/processed/tbill_clean.csv', index=False)

print(f'Clean shape: {df.shape}')
print(f'Date range : {df["auctionDate"].min().date()} → {df["auctionDate"].max().date()}')
print(f'\nTenor breakdown:')
print(df['tenor_days'].value_counts().sort_index())
df[['auctionDate','tenor_days','yield_pct','subscription_ratio','allotment_ratio']].head(6)

## 5. Descriptive Statistics

In [ ]:
desc = df.groupby('tenor_days')['yield_pct'].describe().round(4)
desc.index = ['91-Day', '182-Day', '364-Day']
print('=== NTB Yield (%) Summary by Tenor ===')
print(desc.to_string())

print('\n=== Subscription Ratio Summary ===')
print(df.groupby('tenor_days')['subscription_ratio'].describe().round(3).to_string())

## 6. Yield Trends Over Time (All Tenors)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for tenor, grp in df.groupby('tenor_days'):
    ax.plot(grp['auctionDate'], grp['yield_pct'],
            label=f'{tenor}-Day', alpha=0.85, linewidth=1.2,
            color=COLORS[str(tenor)])

# Shade key monetary policy periods
periods = [
    ('2008-09-01', '2009-06-01', 'Global Financial Crisis'),
    ('2016-01-01', '2017-06-01', 'FX Crisis / Recession'),
    ('2020-03-01', '2021-06-01', 'COVID-19'),
    ('2022-05-01', '2023-12-01', 'CBN Rate Hike Cycle'),
]
for start, end, label in periods:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.08, color='red', label=label)

ax.set_title('Nigerian Treasury Bill Yield (%) — 2002 to 2026', fontsize=14, fontweight='bold')
ax.set_xlabel('Auction Date')
ax.set_ylabel('Yield (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/fig_yield_trend.png', dpi=150)
plt.show()

## 7. Yield Distribution by Tenor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, (tenor, grp) in zip(axes, df.groupby('tenor_days')):
    ax.hist(grp['yield_pct'], bins=30, color=COLORS[str(tenor)],
            edgecolor='white', alpha=0.85)
    ax.axvline(grp['yield_pct'].mean(),   color='black', linewidth=1.5,
               linestyle='--', label=f'Mean: {grp["yield_pct"].mean():.2f}%')
    ax.axvline(grp['yield_pct'].median(), color='gray',  linewidth=1.5,
               linestyle=':',  label=f'Median: {grp["yield_pct"].median():.2f}%')
    ax.set_title(f'{tenor}-Day NTB', fontweight='bold')
    ax.set_xlabel('Yield (%)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('Yield Distribution by Tenor', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_yield_dist.png', dpi=150)
plt.show()

## 8. Yield by Year (Box Plot — Seasonality / Regime Changes)

In [ ]:
df['year'] = df['auctionDate'].dt.year

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

for ax, (tenor, grp) in zip(axes, df.groupby('tenor_days')):
    yearly = [grp[grp['year'] == yr]['yield_pct'].values
              for yr in sorted(grp['year'].unique())]
    years  = sorted(grp['year'].unique())

    bp = ax.boxplot(yearly, patch_artist=True, notch=False,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch in bp['boxes']:
        patch.set_facecolor(COLORS[str(tenor)])
        patch.set_alpha(0.6)

    ax.set_xticks(range(1, len(years)+1))
    ax.set_xticklabels(years, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{tenor}-Day NTB — Annual Yield Distribution', fontweight='bold')
    ax.set_ylabel('Yield (%)')

plt.suptitle('Annual NTB Yield Distributions by Tenor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_annual_boxplot.png', dpi=150)
plt.show()

## 9. Term Structure — Yield Curve Spread

In [ ]:
# Pivot to get all tenors on same auction dates
pivot = df.pivot_table(index='auctionDate', columns='tenor_days', values='yield_pct')
pivot.columns = ['yield_91', 'yield_182', 'yield_364']
pivot = pivot.dropna()

pivot['spread_364_91']  = pivot['yield_364'] - pivot['yield_91']
pivot['spread_182_91']  = pivot['yield_182'] - pivot['yield_91']
pivot['spread_364_182'] = pivot['yield_364'] - pivot['yield_182']

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# All three yields together
for col, label, color in [
    ('yield_91',  '91-Day',  COLORS['91']),
    ('yield_182', '182-Day', COLORS['182']),
    ('yield_364', '364-Day', COLORS['364']),
]:
    axes[0].plot(pivot.index, pivot[col], label=label, color=color, linewidth=1.2)
axes[0].set_title('NTB Yield Curve — All Tenors', fontweight='bold')
axes[0].set_ylabel('Yield (%)')
axes[0].legend()

# Spreads
axes[1].plot(pivot.index, pivot['spread_364_91'],  label='364-91 Day Spread', color='purple', linewidth=1.2)
axes[1].plot(pivot.index, pivot['spread_182_91'],  label='182-91 Day Spread', color='darkorange', linewidth=1.2, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].fill_between(pivot.index, pivot['spread_364_91'], 0,
                     where=pivot['spread_364_91'] > 0, alpha=0.1, color='green', label='Normal (upward slope)')
axes[1].fill_between(pivot.index, pivot['spread_364_91'], 0,
                     where=pivot['spread_364_91'] < 0, alpha=0.1, color='red',   label='Inverted curve')
axes[1].set_title('Term Structure Spread (basis points proxy)', fontweight='bold')
axes[1].set_ylabel('Spread (%)')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.savefig('../data/processed/fig_term_structure.png', dpi=150)
plt.show()

print(f'Inverted curve periods (364<91): {(pivot["spread_364_91"] < 0).sum()} auctions')
print(f'Normal curve (364>91):           {(pivot["spread_364_91"] > 0).sum()} auctions')

## 10. Auction Demand Analysis (Subscription Ratio)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Subscription ratio over time
for tenor, grp in df.groupby('tenor_days'):
    valid = grp[grp['subscription_ratio'].notna() & (grp['subscription_ratio'] < 50)]
    axes[0].plot(valid['auctionDate'], valid['subscription_ratio'],
                 label=f'{tenor}-Day', alpha=0.7, linewidth=1,
                 color=COLORS[str(tenor)])
axes[0].axhline(1, color='black', linewidth=1, linestyle='--', label='Fully subscribed (1x)')
axes[0].set_title('Subscription Ratio Over Time (Total Bids / Amount Offered)', fontweight='bold')
axes[0].set_ylabel('Subscription Ratio (x)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 20)

# Scatter: does high demand correlate with lower yields?
for tenor, grp in df.groupby('tenor_days'):
    valid = grp[grp['subscription_ratio'].notna() & (grp['subscription_ratio'] < 30)]
    axes[1].scatter(valid['subscription_ratio'], valid['yield_pct'],
                    alpha=0.3, s=15, label=f'{tenor}-Day', color=COLORS[str(tenor)])

axes[1].set_title('Yield vs Subscription Ratio (Demand Pressure)', fontweight='bold')
axes[1].set_xlabel('Subscription Ratio (x)')
axes[1].set_ylabel('Yield (%)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/fig_demand_analysis.png', dpi=150)
plt.show()

## 11. Correlation Heatmap

In [ ]:
# Build feature-rich correlation frame
df364 = df[df['tenor_days'] == 364].copy().sort_values('auctionDate')
df364 = df364.merge(pivot[['yield_91','yield_182','spread_364_91','spread_182_91']], on='auctionDate', how='left')

y = df364['yield_pct']
for lag in [1, 2, 3, 6]:
    df364[f'yield_lag_{lag}'] = y.shift(lag)
df364['roll_mean_3'] = y.rolling(3).mean()
df364['roll_std_3']  = y.rolling(3).std()
df364['momentum_1']  = y.diff(1)
df364['year']        = df364['auctionDate'].dt.year
df364['month']       = df364['auctionDate'].dt.month

corr_cols = [
    'yield_pct', 'yield_lag_1', 'yield_lag_2', 'yield_lag_3', 'yield_lag_6',
    'roll_mean_3', 'roll_std_3', 'momentum_1',
    'subscription_ratio', 'allotment_ratio',
    'spread_364_91', 'spread_182_91',
    'yield_91', 'yield_182',
    'month', 'year'
]
corr_df = df364[corr_cols].dropna()

fig, ax = plt.subplots(figsize=(13, 10))
corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Matrix — 364-Day NTB', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_correlation.png', dpi=150)
plt.show()

print('\nTop correlations with yield_pct:')
print(corr_matrix['yield_pct'].drop('yield_pct').sort_values(ascending=False).round(3).to_string())

## 12. Autocorrelation — Is Past Yield Predictive?

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for row, (tenor, grp) in enumerate(df.groupby('tenor_days')):
    series = grp.set_index('auctionDate')['yield_pct'].dropna()
    plot_acf(series,  lags=24, ax=axes[row][0], color=COLORS[str(tenor)],
             title=f'{tenor}-Day — ACF (Autocorrelation)')
    plot_pacf(series, lags=24, ax=axes[row][1], color=COLORS[str(tenor)],
              title=f'{tenor}-Day — PACF (Partial Autocorrelation)', method='ywm')

plt.suptitle('Autocorrelation Analysis — NTB Yields', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_autocorrelation.png', dpi=150)
plt.show()

## 13. Monthly Seasonality

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, (tenor, grp) in zip(axes, df.groupby('tenor_days')):
    monthly = grp.groupby('month')['yield_pct'].mean()
    ax.bar(monthly.index, monthly.values,
           color=COLORS[str(tenor)], alpha=0.8, edgecolor='white')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names, rotation=45)
    ax.set_title(f'{tenor}-Day — Avg Yield by Month', fontweight='bold')
    ax.set_ylabel('Avg Yield (%)')

plt.suptitle('Monthly Seasonality in NTB Yields', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_seasonality.png', dpi=150)
plt.show()

## 14. Rolling Volatility (12-Auction Window)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for tenor, grp in df.groupby('tenor_days'):
    vol = grp.set_index('auctionDate')['yield_pct'].rolling(12).std()
    ax.plot(vol.index, vol.values, label=f'{tenor}-Day',
            color=COLORS[str(tenor)], linewidth=1.2)

ax.set_title('Rolling 12-Auction Yield Volatility (Std Dev)', fontweight='bold')
ax.set_ylabel('Volatility (%)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig_volatility.png', dpi=150)
plt.show()

## 15. EDA Summary — Key Findings

In [ ]:
print('=' * 65)
print('EDA SUMMARY — NIGERIAN TREASURY BILL DATA (2002–2026)')
print('=' * 65)
print(f"""
DATASET
  Total clean records     : {len(df):,}
  Date span               : {df['auctionDate'].min().date()} → {df['auctionDate'].max().date()}
  Tenors                  : 91-Day, 182-Day, 364-Day

YIELD STATISTICS
  91-Day  — Mean: {df[df['tenor_days']==91]['yield_pct'].mean():.2f}%  
             Std:  {df[df['tenor_days']==91]['yield_pct'].std():.2f}%
             Range: {df[df['tenor_days']==91]['yield_pct'].min():.2f}% – {df[df['tenor_days']==91]['yield_pct'].max():.2f}%

  182-Day — Mean: {df[df['tenor_days']==182]['yield_pct'].mean():.2f}%  
             Std:  {df[df['tenor_days']==182]['yield_pct'].std():.2f}%
             Range: {df[df['tenor_days']==182]['yield_pct'].min():.2f}% – {df[df['tenor_days']==182]['yield_pct'].max():.2f}%

  364-Day — Mean: {df[df['tenor_days']==364]['yield_pct'].mean():.2f}%  
             Std:  {df[df['tenor_days']==364]['yield_pct'].std():.2f}%
             Range: {df[df['tenor_days']==364]['yield_pct'].min():.2f}% – {df[df['tenor_days']==364]['yield_pct'].max():.2f}%

KEY FINDINGS FOR MODELLING
  1. Strong autocorrelation at lags 1–6 → lag features are powerful predictors
  2. Term structure spread (364-91) is significant and inverts during crises
  3. Subscription ratio captures demand pressure (market sentiment signal)
  4. Regime changes are visible: 2009, 2016, 2020-2021, 2022-2023
  5. trueYield is 0 for all post-2020 data → use `rate` column as target
  6. Monthly seasonality is mild — Q1 tends slightly higher
  7. Volatility spikes during CBN tightening cycles
""")
print('Saved charts to data/processed/fig_*.png')